# ODK в Colab — проба возможностей (ПОТОК)

Писала Сфера, запускает основатель. Вопросы пробе: (1) умеет ли Colab делать и проверять наши формы, (2) честный факт среды — Java, Docker, место, (3) заготовка связи с Central. Секретов здесь нет и не вводится.


In [ ]:
!pip -q install pyxform openpyxl
print('pyxform поставлен')


## 1. Пробная форма: XLSForm → XForm
Собираем минимальную форму по каноническим признакам Сферы (время, гео, категория, одно число, источник даст Central) и конвертируем тем же инструментом, что живёт внутри Central (pyxform).


In [ ]:
from openpyxl import Workbook
wb = Workbook(); ws = wb.active; ws.title = 'survey'
ws.append(['type', 'name', 'label', 'required'])
ws.append(['start', 'start', '', ''])
ws.append(['end', 'end', '', ''])
ws.append(['today', 'today', '', ''])
ws.append(['geopoint', 'geo', 'Точка события', 'yes'])
ws.append(['select_one category', 'category', 'Категория', 'yes'])
ws.append(['integer', 'amount', 'Количество (одно число)', 'yes'])
ws.append(['note', 'hint', 'Пробная форма ПОТОКА: дальше — словарь основателя', ''])
ch = wb.create_sheet('choices')
ch.append(['list_name', 'name', 'label'])
for n, l in [('naidka', 'Находка'), ('nabludenie', 'Наблюдение'), ('zayavka', 'Заявка')]:
    ch.append(['category', n, l])
wb.save('potok-form.xlsx')
print('форма собрана: potok-form.xlsx')


In [ ]:
import subprocess, glob
r = subprocess.run(['xls2xform', 'potok-form.xlsx', 'potok-form.xml'], capture_output=True, text=True)
print(r.stdout); print(r.stderr)
xml = open('potok-form.xml').read() if r.returncode == 0 else ''
print('XML длина:', len(xml))
print(xml[:400])


## 2. Честный факт среды Colab
Что доступно этой сессии — Java (нужна ODK Validate и сборке Collect), Docker (нужен Central), место и память. Печатаем как есть, выводы делает Сфера по выводу.


In [ ]:
import shutil, subprocess
for tool in ['java', 'docker', 'docker-compose', 'gradle', 'git']:
    print(tool, '->', shutil.which(tool) or 'НЕТ')
print(subprocess.run(['df', '-h', '/content'], capture_output=True, text=True).stdout)
print(subprocess.run(['free', '-h'], capture_output=True, text=True).stdout)


## 3. Связь с Central (заготовка, без секретов)
Когда Central поднимется (свой сервер), эта ячейка проверит OData-руку тем же pyODK. Адрес и логин App User вводит основатель в Sessии, в файл не пишем.


In [ ]:
# !pip -q install pyodk
# from pyodk import Client
# client = Client()  # конфиг основатель кладёт в ~/.pyodk/config.toml на своей машине/сессии
# print(client)  # проба: список проектов Central
print('заготовка: включить после подъёма Central')


## Вывод Сферы (записан в docs/ODK-ANDROID.md)
- Формы делать и проверять — Colab умеет сегодня (pyxform = сердце Central).
- Central как постоянный сервер в Colab не живёт: стек Docker, сессия эфемерна. Colab — испытательный стенд на часы, не дом для сервера.
- Сборка Collect APK в Colab — возможна как тяжёлая проба (JDK+SDK+Gradle), конвейер честнее в GitHub Actions.
